# Notebook 02: Data Cleaning and Harmonisation
## Forecasting Education Performance -- Tanzania BEST Datasets (2020-2024)
**Author:** Habil Masawika | **Project:** Tanzania BEST ML Forecasting

---

### Objectives
1. Extract all relevant indicators from the five BEST workbooks using the `BESTLoader` pipeline
2. Harmonise column names, region names, and data types across years
3. Handle missing values, impossible values, and 2020's infrastructure data gap
4. Derive initial computed indicators (PTR, qualified teacher ratio, etc.)
5. Produce and save the cleaned analytical panel dataset
6. Generate a comprehensive data quality report

### Methodology
Extraction follows a pattern-based approach: each sheet type has a dedicated extractor
function that reads the raw Excel with `header=None`, identifies data rows by matching
the first column against a canonical list of 26 Tanzania regions, and selects columns
by known position. This makes the pipeline robust to the merged-cell and multi-row header
structures common in administrative data workbooks.

In [ ]:
import sys, warnings
sys.path.insert(0, '/home/claude/BEST-ML-Forecasting/src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utilities import setup_logging, set_seeds, ProjectPaths, Timer, save_dataframe
from data_loader import BESTLoader, load_best_data
from preprocessing import BESTCleaner, missing_summary
from visualization import plot_missing_heatmap

set_seeds(42)
logger = setup_logging()
paths  = ProjectPaths()
print("Modules loaded successfully.")

In [ ]:
# ── Step 1: Extract raw panel ────────────────────────────────────────────
with Timer("Full data extraction"):
    loader = BESTLoader(data_dir="/mnt/user-data/uploads")
    raw_panel = loader.build_regional_panel()

print(f"Raw panel: {raw_panel.shape[0]} rows x {raw_panel.shape[1]} columns")
print(f"Years: {sorted(raw_panel['year'].unique())}")
print(f"Regions: {raw_panel['region'].nunique()}")
print()
print("Columns:")
for c in raw_panel.columns:
    print(f"  {c}: {raw_panel[c].dtype}")

In [ ]:
# ── Step 2: Missing value audit (pre-cleaning) ───────────────────────────
miss_pre = missing_summary(raw_panel)
print("Missing values BEFORE cleaning:")
print(miss_pre.head(20).to_string(index=False))

In [ ]:
# ── Step 3: Run cleaning pipeline ───────────────────────────────────────
with Timer("Cleaning pipeline"):
    cleaner = BESTCleaner(raw_panel, log_transforms=True)
    panel   = cleaner.run()

print("\nTransformation log:")
for entry in cleaner.get_transform_log():
    print(f"  [+] {entry}")

In [ ]:
# ── Step 4: Missing value audit (post-cleaning) ──────────────────────────
miss_post = missing_summary(panel)
print("Missing values AFTER cleaning:")
print(miss_post.to_string(index=False))

In [ ]:
# ── Step 5: Data quality report ──────────────────────────────────────────
quality = cleaner.data_quality_report()
print("Data quality summary (numeric columns):")
print(quality.to_string(index=False))

In [ ]:
# ── Step 6: Missing data visualisation ───────────────────────────────────
fig = plot_missing_heatmap(panel,
      save_path=paths.fig('nb02_missing_heatmap.png'))
plt.show()
print("Missing data heatmap saved.")

In [ ]:
# ── Step 7: Sanity checks ────────────────────────────────────────────────
checks = {
    'CSEE pass rate in [40,100]':      ((panel['csee_pass_rate'] >= 40) & (panel['csee_pass_rate'] <= 100)).all(),
    'Electricity pct in [0,100]':      ((panel['pct_schools_electricity'] >= 0) & (panel['pct_schools_electricity'] <= 100)).mean() > 0.95,
    'PTR regional positive':           (panel['ptr_regional'] > 0).mean() > 0.9,
    'Total schools positive':          (panel['total_schools'] > 0).all(),
    'No duplicate (year, region)':     panel.duplicated(subset=['year','region']).sum() == 0,
    'All 26 regions present (2022)':   panel[panel['year']==2022]['region'].nunique() == 26,
}
print("Sanity checks:")
for label, result in checks.items():
    status = 'PASS' if result else 'FAIL'
    print(f"  [{status}] {label}")

In [ ]:
# ── Step 8: Panel overview ───────────────────────────────────────────────
key_cols = ['csee_pass_rate','total_schools','enrolment_f1f4','total_teachers',
            'pct_schools_electricity','desktops_per_school','ptr_regional',
            'ptr_national','qualified_teacher_ratio','dropout_rate_pct',
            'gross_completion_rate','teachers_per_school','nongovt_share']
key_cols = [c for c in key_cols if c in panel.columns]
print("Descriptive statistics (key indicators):")
print(panel[key_cols].describe().round(2).to_string())

In [ ]:
# ── Step 9: Sample rows ──────────────────────────────────────────────────
print("Sample panel rows (first 8):")
display_cols = ['year','region','total_schools','enrolment_f1f4',
                'total_teachers','ptr_regional','pct_schools_electricity',
                'csee_pass_rate']
display_cols = [c for c in display_cols if c in panel.columns]
print(panel[display_cols].head(8).to_string(index=False))

In [ ]:
# ── Step 10: Save cleaned datasets ──────────────────────────────────────
# Main panel
save_dataframe(panel, paths.processed('best_panel_cleaned.csv'))

# Also save the national CSEE time series separately
csee_national = loader.extract_csee_national()
save_dataframe(csee_national, paths.processed('csee_national_trend.csv'))

# Dropout national
dropout_nat = loader.extract_dropout_national()
save_dataframe(dropout_nat, paths.processed('dropout_national.csv'))

# Completion rate
completion   = loader.extract_completion_rate()
save_dataframe(completion, paths.processed('completion_rate.csv'))

print("\nAll cleaned datasets saved to data/processed/")
print(f"  Panel shape:      {panel.shape}")
print(f"  CSEE years:       {len(csee_national)}")
print(f"  Checksum (panel): ...")
from utilities import dataframe_checksum
print(f"  Checksum (panel): {dataframe_checksum(panel)}")